# Warehouse Sizer — Cost Optimizer

This notebook analyzes virtual warehouse utilization from **ACCOUNT_USAGE** views and generates **AI-powered sizing recommendations** using Snowflake Cortex LLM functions.

### Data Sources
- `SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_LOAD_HISTORY` — 5-minute interval load metrics (AVG_RUNNING, AVG_QUEUED_LOAD)
- `SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY` — hourly credit consumption
- `SHOW WAREHOUSES` — current warehouse sizes

### Cortex AI
- `SNOWFLAKE.CORTEX.COMPLETE` (llama3.1-70b) — generates sizing recommendations per warehouse

In [ ]:
%%sql -r context_setup
-- Set execution context for the notebook.
-- This enables cell referencing and interactive datagrids in Python cells.
USE DATABASE COST_MANAGEMENT;
USE SCHEMA PUBLIC;

In [ ]:
##############################################################################
# Configuration — User-Adjustable Parameters
# These thresholds control how the Cortex AI model decides sizing actions.
# Adjust the date range to focus on a specific analysis window.
# The queued load threshold and utilization floor are passed to the LLM prompt.
##############################################################################
from datetime import date, timedelta
import json
import pandas as pd

# --- Date Range (last 30 days by default) ---
END_DATE = date.today()
START_DATE = END_DATE - timedelta(days=30)

# --- Queued Load Threshold ---
# If a warehouse's avg queued load exceeds this, Cortex AI recommends sizing UP
# because queries are waiting for compute resources.
QUEUED_THRESHOLD = 0.05

# --- Utilization Floor (%) ---
# If avg running load is below this % with near-zero queuing, the warehouse is
# over-provisioned and Cortex AI recommends sizing DOWN.
UTILIZATION_FLOOR = 10

print(f"Analysis window: {START_DATE} to {END_DATE}")
print(f"Queued Load Threshold: {QUEUED_THRESHOLD}")
print(f"Utilization Floor: {UTILIZATION_FLOOR}%")

## Step 1: Query Warehouse Load History

Fetch 5-minute interval load metrics from `SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_LOAD_HISTORY`.
- **AVG_RUNNING**: ratio of query execution time to interval time — shows how busy the warehouse is.
- **AVG_QUEUED_LOAD**: ratio of queued time to interval time — indicates queries waiting for compute.

In [ ]:
%%sql -r load_history_df
-- Warehouse Load History: 5-minute interval load metrics
-- AVG_RUNNING shows compute utilization; AVG_QUEUED_LOAD shows queuing pressure.
-- These are the core inputs for sizing analysis.
SELECT
    WAREHOUSE_NAME,
    START_TIME,
    AVG_RUNNING,
    AVG_QUEUED_LOAD
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_LOAD_HISTORY
WHERE START_TIME >= DATEADD('day', -30, CURRENT_DATE())
  AND START_TIME <= CURRENT_DATE()
ORDER BY WAREHOUSE_NAME, START_TIME

## Step 2: Query Warehouse Metering History

Fetch total credit consumption per warehouse from `SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY`.
**CREDITS_USED** is the primary cost metric — it determines how much each warehouse costs.

In [ ]:
%%sql -r metering_df
-- Warehouse Metering History: total credits consumed per warehouse
-- CREDITS_USED drives cost ranking — most expensive warehouses are analyzed first.
SELECT
    WAREHOUSE_NAME,
    SUM(CREDITS_USED) AS TOTAL_CREDITS_USED
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY
WHERE START_TIME >= DATEADD('day', -30, CURRENT_DATE())
  AND START_TIME <= CURRENT_DATE()
GROUP BY WAREHOUSE_NAME

## Step 3: Get Current Warehouse Sizes

Run `SHOW WAREHOUSES` to retrieve each warehouse's current size configuration.
This is needed so the Cortex AI recommendation can suggest the next size up or down.

In [ ]:
%%sql -r show_wh_raw
-- Current warehouse sizes from SHOW WAREHOUSES
-- Needed to map current size to the next size up/down in recommendations.
SHOW WAREHOUSES

In [ ]:
%%sql -r warehouse_sizes_df
-- Extract warehouse name and size from the SHOW WAREHOUSES result
SELECT "name" AS WAREHOUSE_NAME, "size" AS WAREHOUSE_SIZE
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))

## Step 4: Build Summary DataFrame

Aggregate per-warehouse metrics from load history and join with metering (credit) data and current sizes. This produces one row per warehouse with all metrics needed for the summary table and Cortex AI recommendation prompt.

In [ ]:
##############################################################################
# Build Summary DataFrame
# Aggregate per-warehouse metrics from the 5-minute load history intervals,
# then join with credit consumption and current warehouse sizes.
# Limit to TOP 3 warehouses by credits for faster Cortex AI processing.
##############################################################################

load_df = load_history_df.copy()

if load_df.empty:
    raise ValueError("No warehouse load data found for the selected date range.")

# Compute per-warehouse aggregates from 5-minute intervals.
agg_df = (
    load_df.groupby("WAREHOUSE_NAME")
    .agg(
        AVG_RUNNING=("AVG_RUNNING", "mean"),
        AVG_QUEUED_LOAD=("AVG_QUEUED_LOAD", "mean"),
        PEAK_RUNNING=("AVG_RUNNING", "max"),
    )
    .reset_index()
)

# Build sparkline data: collect AVG_RUNNING values over time as a list per warehouse.
sparkline_data = (
    load_df.sort_values("START_TIME")
    .groupby("WAREHOUSE_NAME")["AVG_RUNNING"]
    .apply(list)
    .reset_index()
    .rename(columns={"AVG_RUNNING": "LOAD_TREND"})
)

# Also keep the raw time-series for per-warehouse sparkline charts.
time_series_data = (
    load_df.sort_values("START_TIME")
    .groupby("WAREHOUSE_NAME")[["START_TIME", "AVG_RUNNING", "AVG_QUEUED_LOAD"]]
    .apply(lambda g: g.reset_index(drop=True))
)

# Merge all data sources into one summary DataFrame.
summary_df = agg_df.merge(metering_df, on="WAREHOUSE_NAME", how="left")
summary_df = summary_df.merge(warehouse_sizes_df, on="WAREHOUSE_NAME", how="left")
summary_df = summary_df.merge(sparkline_data, on="WAREHOUSE_NAME", how="left")

summary_df["TOTAL_CREDITS_USED"] = summary_df["TOTAL_CREDITS_USED"].fillna(0)
summary_df["WAREHOUSE_SIZE"] = summary_df["WAREHOUSE_SIZE"].fillna("Unknown")

# Sort by credits used descending, then take TOP 3 for speed.
summary_df = summary_df.sort_values("TOTAL_CREDITS_USED", ascending=False).reset_index(drop=True)
summary_df = summary_df.head(3).reset_index(drop=True)

# Filter load_df to only include our top 3 warehouses.
top_wh_names = summary_df["WAREHOUSE_NAME"].tolist()
load_df = load_df[load_df["WAREHOUSE_NAME"].isin(top_wh_names)]

print(f"Analyzing top {len(summary_df)} warehouses by credit consumption:")
for _, r in summary_df.iterrows():
    print(f"  {r['WAREHOUSE_NAME']} ({r['WAREHOUSE_SIZE']}) — {r['TOTAL_CREDITS_USED']:.1f} credits")

## Step 5: Cortex AI Sizing Recommendations

For each warehouse, call `SNOWFLAKE.CORTEX.COMPLETE` with the **llama3.1-70b** model. The prompt includes all relevant metrics plus the user-configured thresholds so the model can make context-aware sizing decisions.

The LLM returns structured JSON with:
- **action**: `SIZE UP`, `SIZE DOWN`, or `KEEP`
- **recommendation**: one-sentence summary
- **rationale**: 2-3 sentence explanation
- **sql**: the `ALTER WAREHOUSE` statement if a change is needed

In [ ]:
##############################################################################
# Cortex AI Sizing Recommendations
# For each warehouse, call SNOWFLAKE.CORTEX.COMPLETE (llama3.1-70b) with a
# prompt containing all metrics and thresholds. The system prompt instructs
# the LLM to act as a Snowflake optimization expert and return structured
# JSON with the action, recommendation, rationale, and SQL.
##############################################################################
import re
from snowflake.snowpark.context import get_active_session

session = get_active_session()

SYSTEM_PROMPT = """You are a Snowflake warehouse optimization expert. Given the warehouse metrics below, determine if the warehouse should be sized UP, sized DOWN, or kept the same. Provide:
1. A one-sentence recommendation (e.g. SIZE DOWN for {warehouse_name}).
2. A brief rationale (2-3 sentences max).
3. The exact ALTER WAREHOUSE SQL command if a change is needed, using the fully qualified warehouse name.
Rules:
- If avg_queued_load > threshold, recommend sizing UP.
- If avg_running < utilization_floor AND avg_queued_load is near 0, recommend sizing DOWN.
- Map sizes: XS < S < M < L < XL < 2XL < 3XL < 4XL.
- Never recommend below X-Small.
Respond in JSON with keys: action, recommendation, rationale, sql"""


def parse_recommendation(raw_response):
    """Parse the JSON response from Cortex COMPLETE.
    The LLM may wrap JSON in markdown code fences, so we strip those.
    Uses regex to extract the JSON object between curly braces."""
    try:
        # Try to extract JSON object using regex — handles code fences,
        # extra text before/after, and various fence styles (``` or ```json).
        match = re.search(r'\{[^{}]*\}', raw_response, re.DOTALL)
        if match:
            return json.loads(match.group())
        # Fallback: try parsing the whole string
        return json.loads(raw_response.strip())
    except (json.JSONDecodeError, IndexError, AttributeError):
        return {
            "action": "keep",
            "recommendation": "Unable to parse recommendation.",
            "rationale": raw_response[:200] if raw_response else "No response",
            "sql": "",
        }


recommendations = {}
for idx, row in summary_df.iterrows():
    wh_name = row["WAREHOUSE_NAME"]
    
    # Build the user prompt with all warehouse metrics for the LLM.
    user_prompt = (
        f"Warehouse: {wh_name}\n"
        f"Current Size: {row['WAREHOUSE_SIZE']}\n"
        f"Avg Running: {float(row['AVG_RUNNING']):.4f}\n"
        f"Avg Queued Load: {float(row['AVG_QUEUED_LOAD']):.4f}\n"
        f"Peak Running: {float(row['PEAK_RUNNING']):.4f}\n"
        f"Total Credits Used: {float(row['TOTAL_CREDITS_USED']):.2f}\n"
        f"Queued Load Threshold: {QUEUED_THRESHOLD}\n"
        f"Utilization Floor: {UTILIZATION_FLOOR}%"
    )

    # Escape single quotes for safe SQL embedding.
    safe_system = SYSTEM_PROMPT.replace("'", "\\'")
    safe_user = user_prompt.replace("'", "\\'")

    # Call SNOWFLAKE.CORTEX.COMPLETE with llama3.1-70b
    query = f"""
        SELECT SNOWFLAKE.CORTEX.COMPLETE(
            'llama3.1-70b',
            CONCAT('{safe_system}', '\n\nMetrics:\n', '{safe_user}')
        ) AS RECOMMENDATION
    """
    result = session.sql(query).to_pandas()
    raw = result["RECOMMENDATION"].iloc[0] if not result.empty else "{}"
    recommendations[wh_name] = parse_recommendation(raw)
    print(f"  [{idx+1}/{len(summary_df)}] {wh_name}: {recommendations[wh_name].get('action', 'N/A')}")

print(f"\nCompleted recommendations for {len(recommendations)} warehouses.")

In [ ]:
##############################################################################
# Classify Recommendations & Render Filter Toolbar
# Emulates the Snowflake Warehouse Sizing tool's filter dropdown with
# category badges: All, Size down, Size up, Keep — each with counts
# and descriptive text, rendered as a matplotlib figure.
##############################################################################
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.offsetbox import AnchoredText
import numpy as np

def classify_action(action_str):
    lower = action_str.lower().strip()
    if "up" in lower:
        return "size_up"
    elif "down" in lower:
        return "size_down"
    else:
        return "keep"

summary_df["ACTION"] = summary_df["WAREHOUSE_NAME"].apply(
    lambda wh: classify_action(recommendations.get(wh, {}).get("action", "keep"))
)
summary_df["RECOMMENDATION"] = summary_df["WAREHOUSE_NAME"].apply(
    lambda wh: recommendations.get(wh, {}).get("recommendation", "N/A")
)
summary_df["RATIONALE"] = summary_df["WAREHOUSE_NAME"].apply(
    lambda wh: recommendations.get(wh, {}).get("rationale", "N/A")
)
summary_df["SQL"] = summary_df["WAREHOUSE_NAME"].apply(
    lambda wh: recommendations.get(wh, {}).get("sql", "")
)

count_all = len(summary_df)
count_down = len(summary_df[summary_df["ACTION"] == "size_down"])
count_up = len(summary_df[summary_df["ACTION"] == "size_up"])
count_keep = len(summary_df[summary_df["ACTION"] == "keep"])

# --- Render filter toolbar as a figure ---
categories = [
    (f"All ({count_all})", "Show all insights", "#E8EAF6", "#333"),
    (f"Size down ({count_down})", "Warehouses where most queries are not\nusing all compute power available", "#FFF3E0", "#E65100"),
    (f"Size up ({count_up})", "Warehouses where queries are frequently\nqueuing for compute", "#FFEBEE", "#C62828"),
    (f"Keep ({count_keep})", "Warehouses that are appropriately sized", "#E8F5E9", "#2E7D32"),
]

fig, axes = plt.subplots(1, len(categories), figsize=(14, 1.6))
fig.suptitle("", fontsize=1)
fig.patch.set_facecolor("white")

for ax, (title, desc, bg, fg) in zip(axes, categories):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_facecolor(bg)
    ax.text(0.08, 0.7, title, fontsize=13, fontweight="bold", color=fg,
            va="center", ha="left", transform=ax.transAxes)
    ax.text(0.08, 0.25, desc, fontsize=8, color="#666",
            va="center", ha="left", transform=ax.transAxes, linespacing=1.3)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.patch.set_alpha(0.6)
    # Rounded border effect
    rect = mpatches.FancyBboxPatch((0.02, 0.08), 0.96, 0.84, boxstyle="round,pad=0.05",
                                    facecolor=bg, edgecolor="#ddd", linewidth=1,
                                    transform=ax.transAxes, zorder=0)
    ax.add_patch(rect)

plt.subplots_adjust(wspace=0.15, left=0.01, right=0.99, top=0.95, bottom=0.05)
plt.show()

In [ ]:
##############################################################################
# Warehouse Summary Cards with Sparkline Charts
# Emulates the Snowflake Warehouse Sizing tool's per-warehouse rows.
# Each warehouse gets a visual card showing:
#   - Left: warehouse name, size, action badge
#   - Center: sparkline chart of query load (AVG_RUNNING) over time
#   - Right: key metrics (avg running, avg queued, credits, peak)
##############################################################################
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

ACTION_COLORS = {
    "size_down": ("#E65100", "#FFF3E0", "SIZE DOWN"),
    "size_up":   ("#C62828", "#FFEBEE", "SIZE UP"),
    "keep":      ("#2E7D32", "#E8F5E9", "KEEP"),
}

n_wh = len(summary_df)
fig, axes = plt.subplots(n_wh, 3, figsize=(16, n_wh * 2.8),
                          gridspec_kw={"width_ratios": [2.5, 5, 3]})
fig.patch.set_facecolor("white")

if n_wh == 1:
    axes = axes.reshape(1, -1)

for i, (_, row) in enumerate(summary_df.iterrows()):
    wh_name = row["WAREHOUSE_NAME"]
    wh_size = row["WAREHOUSE_SIZE"]
    action = row["ACTION"]
    fg, bg, label = ACTION_COLORS.get(action, ("#333", "#F5F5F5", "KEEP"))
    
    # --- LEFT PANEL: Name, Size, Badge ---
    ax_left = axes[i, 0]
    ax_left.set_xlim(0, 1)
    ax_left.set_ylim(0, 1)
    ax_left.set_facecolor("white")
    ax_left.text(0.05, 0.72, wh_name, fontsize=12, fontweight="bold", color="#1a1a2e",
                 va="center", ha="left", transform=ax_left.transAxes)
    ax_left.text(0.05, 0.48, f"Size: {wh_size}", fontsize=10, color="#555",
                 va="center", ha="left", transform=ax_left.transAxes)
    # Action badge
    badge = mpatches.FancyBboxPatch((0.05, 0.08), 0.55, 0.25,
                                     boxstyle="round,pad=0.03",
                                     facecolor=bg, edgecolor=fg, linewidth=1.5,
                                     transform=ax_left.transAxes)
    ax_left.add_patch(badge)
    ax_left.text(0.32, 0.20, label, fontsize=9, fontweight="bold", color=fg,
                 va="center", ha="center", transform=ax_left.transAxes)
    for spine in ax_left.spines.values():
        spine.set_visible(False)
    ax_left.set_xticks([])
    ax_left.set_yticks([])
    
    # --- CENTER PANEL: Load Over Time Sparkline ---
    ax_center = axes[i, 1]
    trend = row["LOAD_TREND"]
    if trend and len(trend) > 1:
        x = np.arange(len(trend))
        y = np.array(trend, dtype=float)
        ax_center.fill_between(x, y, alpha=0.15, color="#4285F4")
        ax_center.plot(x, y, color="#4285F4", linewidth=1.2)
        ax_center.set_xlim(0, len(trend) - 1)
        ax_center.set_ylim(0, max(y.max() * 1.2, 0.01))
    ax_center.set_facecolor("#FAFAFA")
    ax_center.set_title("Query Load Over Time", fontsize=8, color="#999", loc="left", pad=2)
    ax_center.tick_params(axis="both", labelsize=7, colors="#aaa")
    ax_center.spines["top"].set_visible(False)
    ax_center.spines["right"].set_visible(False)
    ax_center.spines["bottom"].set_color("#ddd")
    ax_center.spines["left"].set_color("#ddd")
    
    # --- RIGHT PANEL: Key Metrics ---
    ax_right = axes[i, 2]
    ax_right.set_xlim(0, 1)
    ax_right.set_ylim(0, 1)
    ax_right.set_facecolor("white")
    
    metrics = [
        ("Avg Running", f"{float(row['AVG_RUNNING']):.4f}"),
        ("Avg Queued", f"{float(row['AVG_QUEUED_LOAD']):.4f}"),
        ("Credits Used", f"{float(row['TOTAL_CREDITS_USED']):.1f}"),
        ("Peak Running", f"{float(row['PEAK_RUNNING']):.4f}"),
    ]
    for j, (mlabel, mval) in enumerate(metrics):
        y_pos = 0.82 - j * 0.22
        ax_right.text(0.08, y_pos, mlabel, fontsize=8, color="#888",
                      va="center", ha="left", transform=ax_right.transAxes)
        ax_right.text(0.92, y_pos, mval, fontsize=10, fontweight="bold", color="#1a1a2e",
                      va="center", ha="right", transform=ax_right.transAxes)
    
    for spine in ax_right.spines.values():
        spine.set_visible(False)
    ax_right.set_xticks([])
    ax_right.set_yticks([])
    
    # Card border for each row
    for ax in [ax_left, ax_center, ax_right]:
        rect = mpatches.FancyBboxPatch((0, 0), 1, 1, boxstyle="round,pad=0.02",
                                        facecolor="none", edgecolor="#E0E0E0", linewidth=0.8,
                                        transform=ax.transAxes, zorder=-1)
        ax.add_patch(rect)

plt.subplots_adjust(hspace=0.5, wspace=0.08, left=0.02, right=0.98, top=0.96, bottom=0.04)
plt.suptitle("Warehouse Sizing Insights", fontsize=14, fontweight="bold", color="#1a1a2e", y=0.99)
plt.show()

In [ ]:
##############################################################################
# Recommended SQL — Styled Code Panel
# Renders the ALTER WAREHOUSE statements in a matplotlib figure styled
# like a dark code editor panel, matching the modal from the sizing tool.
##############################################################################
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

actionable_df = summary_df[summary_df["ACTION"] != "keep"]
num_actionable = len(actionable_df)

sql_lines = []
for _, row in actionable_df.iterrows():
    action_label = row["ACTION"].replace("_", " ").upper()
    wh_name = row["WAREHOUSE_NAME"]
    sql_stmt = row["SQL"]
    if sql_stmt:
        sql_lines.append(f"-- {action_label} for {wh_name}")
        sql_lines.append(f"{sql_stmt};")
        sql_lines.append("")

if not sql_lines:
    sql_lines = ["-- No sizing changes recommended.", "-- All warehouses are appropriately sized."]

sql_text = "\n".join(sql_lines)

# Render as a dark-themed code block
fig, ax = plt.subplots(figsize=(14, max(2, len(sql_lines) * 0.35)))
fig.patch.set_facecolor("white")
ax.set_facecolor("#1E1E2E")

# Title bar
rect = mpatches.FancyBboxPatch((0.0, 0.88), 1.0, 0.12, boxstyle="round,pad=0.01",
                                facecolor="#2D2D44", edgecolor="none",
                                transform=ax.transAxes, zorder=2)
ax.add_patch(rect)
ax.text(0.02, 0.94, f"Recommended SQL for {num_actionable} Insights",
        fontsize=11, fontweight="bold", color="#E0E0E0",
        va="center", ha="left", transform=ax.transAxes, zorder=3)

# SQL code text
ax.text(0.02, 0.80, sql_text, fontsize=9, fontfamily="monospace", color="#A6E3A1",
        va="top", ha="left", transform=ax.transAxes, linespacing=1.6)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
for spine in ax.spines.values():
    spine.set_color("#333")
ax.set_xticks([])
ax.set_yticks([])

plt.tight_layout()
plt.show()

In [ ]:
##############################################################################
# Detailed Recommendation Cards
# Per-warehouse cards with colored left border, name, action badge,
# recommendation text, rationale, and SQL — emulating UI card components
# from the Snowflake sizing tool.
##############################################################################
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import textwrap

ACTION_STYLES = {
    "size_down": ("#E65100", "#FFF8F0", "SIZE DOWN"),
    "size_up":   ("#C62828", "#FFF5F5", "SIZE UP"),
    "keep":      ("#2E7D32", "#F0FFF0", "KEEP"),
}

n = len(summary_df)
fig, axes = plt.subplots(n, 1, figsize=(14, n * 2.5))
fig.patch.set_facecolor("white")

if n == 1:
    axes = [axes]

for i, (_, row) in enumerate(summary_df.iterrows()):
    ax = axes[i]
    wh_name = row["WAREHOUSE_NAME"]
    action = row["ACTION"]
    rec = recommendations.get(wh_name, {})
    fg, bg, label = ACTION_STYLES.get(action, ("#333", "#F5F5F5", "KEEP"))
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_facecolor(bg)
    
    # Colored left border
    left_bar = mpatches.FancyBboxPatch((0, 0), 0.012, 1.0,
                                        boxstyle="square,pad=0",
                                        facecolor=fg, edgecolor="none",
                                        transform=ax.transAxes, zorder=3)
    ax.add_patch(left_bar)
    
    # Warehouse name + badge
    ax.text(0.04, 0.82, wh_name, fontsize=13, fontweight="bold", color="#1a1a2e",
            va="center", ha="left", transform=ax.transAxes)
    
    badge = mpatches.FancyBboxPatch((0.35, 0.72), 0.12, 0.18,
                                     boxstyle="round,pad=0.02",
                                     facecolor=fg, edgecolor="none",
                                     transform=ax.transAxes, zorder=2)
    ax.add_patch(badge)
    ax.text(0.41, 0.81, label, fontsize=8, fontweight="bold", color="white",
            va="center", ha="center", transform=ax.transAxes, zorder=3)
    
    # Recommendation
    rec_text = rec.get("recommendation", "N/A")
    wrapped = textwrap.fill(rec_text, width=100)
    ax.text(0.04, 0.55, wrapped, fontsize=10, color="#333",
            va="center", ha="left", transform=ax.transAxes)
    
    # Rationale
    rationale = rec.get("rationale", "N/A")
    wrapped_r = textwrap.fill(rationale, width=120)
    ax.text(0.04, 0.25, wrapped_r, fontsize=8.5, color="#666", style="italic",
            va="center", ha="left", transform=ax.transAxes, linespacing=1.3)
    
    # SQL snippet on the right
    sql = rec.get("sql", "")
    if sql:
        ax.text(0.96, 0.15, sql, fontsize=7.5, fontfamily="monospace", color="#4A148C",
                va="center", ha="right", transform=ax.transAxes,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="#F3E5F5", edgecolor="#CE93D8", alpha=0.8))
    
    for spine in ax.spines.values():
        spine.set_color("#E0E0E0")
    ax.set_xticks([])
    ax.set_yticks([])

plt.subplots_adjust(hspace=0.4, left=0.02, right=0.98, top=0.95, bottom=0.05)
plt.suptitle("Detailed Recommendations", fontsize=14, fontweight="bold", color="#1a1a2e", y=0.99)
plt.show()

In [ ]:
##############################################################################
# Estimated Savings from Sizing Recommendations
# Uses Snowflake's published credits/hour rates per warehouse size to
# estimate the credit and cost savings if recommendations are applied.
# Sizes double in credits as they increase: XS=1, S=2, M=4, L=8, etc.
# Default credit price is $3.00/credit (adjust CREDIT_PRICE below).
##############################################################################
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Credits per hour for each warehouse size (Gen1 standard warehouses)
SIZE_CREDITS = {
    "X-Small": 1, "Small": 2, "Medium": 4, "Large": 8,
    "X-Large": 16, "2X-Large": 32, "3X-Large": 64, "4X-Large": 128,
    "5X-Large": 256, "6X-Large": 512,
}

# Ordered list of sizes for stepping up/down
SIZE_ORDER = ["X-Small", "Small", "Medium", "Large", "X-Large",
              "2X-Large", "3X-Large", "4X-Large", "5X-Large", "6X-Large"]

# Adjust this to your contract rate
CREDIT_PRICE = 3.00

def get_recommended_size(current_size, action):
    """Step one size up or down based on the action."""
    if current_size not in SIZE_ORDER:
        return current_size
    idx = SIZE_ORDER.index(current_size)
    if action == "size_down" and idx > 0:
        return SIZE_ORDER[idx - 1]
    elif action == "size_up" and idx < len(SIZE_ORDER) - 1:
        return SIZE_ORDER[idx + 1]
    return current_size

# Calculate savings per warehouse
savings_rows = []
for _, row in summary_df.iterrows():
    wh = row["WAREHOUSE_NAME"]
    current = row["WAREHOUSE_SIZE"]
    action = row["ACTION"]
    credits_used = float(row["TOTAL_CREDITS_USED"])
    
    current_rate = SIZE_CREDITS.get(current, 1)
    new_size = get_recommended_size(current, action)
    new_rate = SIZE_CREDITS.get(new_size, current_rate)
    
    if current_rate > 0:
        ratio = new_rate / current_rate
        estimated_new_credits = credits_used * ratio
        credit_savings = credits_used - estimated_new_credits
        cost_savings = credit_savings * CREDIT_PRICE
    else:
        estimated_new_credits = credits_used
        credit_savings = 0
        cost_savings = 0
    
    savings_rows.append({
        "warehouse": wh,
        "current_size": current,
        "new_size": new_size,
        "action": action,
        "current_credits": credits_used,
        "estimated_credits": estimated_new_credits,
        "credit_savings": credit_savings,
        "cost_savings": cost_savings,
    })

# --- Render Savings Dashboard ---
total_credit_savings = sum(r["credit_savings"] for r in savings_rows)
total_cost_savings = sum(r["cost_savings"] for r in savings_rows)
monthly_projection = total_cost_savings
annual_projection = monthly_projection * 12

fig, (ax_summary, ax_bars) = plt.subplots(2, 1, figsize=(14, 4.5 + len(savings_rows) * 1.2),
                                           gridspec_kw={"height_ratios": [1, len(savings_rows) * 0.8 + 0.5]})
fig.patch.set_facecolor("white")

# --- Summary Cards ---
ax_summary.set_xlim(0, 1)
ax_summary.set_ylim(0, 1)
ax_summary.set_facecolor("#F8F9FA")

card_data = [
    ("Credit Savings (30d)", f"{total_credit_savings:,.1f}", "#4285F4"),
    ("Monthly Savings", f"${monthly_projection:,.2f}", "#34A853"),
    ("Projected Annual Savings", f"${annual_projection:,.2f}", "#34A853"),
]

for k, (clabel, cval, ccolor) in enumerate(card_data):
    cx = 0.05 + k * 0.32
    card_bg = mpatches.FancyBboxPatch((cx, 0.1), 0.28, 0.8,
                                       boxstyle="round,pad=0.03",
                                       facecolor="white", edgecolor="#E0E0E0",
                                       linewidth=1, transform=ax_summary.transAxes)
    ax_summary.add_patch(card_bg)
    ax_summary.text(cx + 0.14, 0.7, clabel, fontsize=9, color="#666",
                    va="center", ha="center", transform=ax_summary.transAxes)
    ax_summary.text(cx + 0.14, 0.35, cval, fontsize=20, fontweight="bold", color=ccolor,
                    va="center", ha="center", transform=ax_summary.transAxes)

for spine in ax_summary.spines.values():
    spine.set_visible(False)
ax_summary.set_xticks([])
ax_summary.set_yticks([])

# --- Per-Warehouse Savings Bars ---
warehouses = [r["warehouse"] for r in savings_rows]
current_credits = [r["current_credits"] for r in savings_rows]
estimated_credits = [r["estimated_credits"] for r in savings_rows]
actions = [r["action"] for r in savings_rows]

y_pos = np.arange(len(warehouses))
bar_height = 0.35

ax_bars.barh(y_pos + bar_height/2, current_credits, bar_height,
             color="#BDBDBD", label="Current Credits (30d)", edgecolor="white")

bar_colors = ["#4285F4" if a == "size_down" else "#EA4335" if a == "size_up" else "#34A853" for a in actions]
ax_bars.barh(y_pos - bar_height/2, estimated_credits, bar_height,
             color=bar_colors, label="Estimated Credits (30d)", edgecolor="white")

for i, r in enumerate(savings_rows):
    if r["credit_savings"] != 0:
        savings_text = f"Save {r['credit_savings']:.1f} credits (${r['cost_savings']:.2f})"
        color = "#34A853" if r["credit_savings"] > 0 else "#EA4335"
    else:
        savings_text = "No change"
        color = "#999"
    ax_bars.text(max(current_credits) * 1.02, i, savings_text,
                 fontsize=9, color=color, va="center", fontweight="bold")
    size_label = f"{r['current_size']}"
    if r['current_size'] != r['new_size']:
        size_label += f" → {r['new_size']}"
    ax_bars.text(-0.3, i, size_label, fontsize=8, color="#666", va="center", ha="right",
                 clip_on=False)

ax_bars.set_yticks(y_pos)
ax_bars.set_yticklabels(warehouses, fontsize=10, fontweight="bold")
ax_bars.invert_yaxis()
ax_bars.set_xlabel("Credits (30-day period)", fontsize=10, color="#666")
ax_bars.spines["top"].set_visible(False)
ax_bars.spines["right"].set_visible(False)
ax_bars.spines["left"].set_visible(False)
ax_bars.tick_params(axis="y", length=0)
ax_bars.legend(loc="lower right", fontsize=9, framealpha=0.9)

plt.suptitle("Estimated Savings from Sizing Recommendations",
             fontsize=14, fontweight="bold", color="#1a1a2e", y=1.02)
plt.subplots_adjust(hspace=0.4, left=0.15, right=0.82, top=0.93, bottom=0.08)
plt.show()